In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import welch, butter, filtfilt

In [ ]:
def butter_highpass_filter(data, cutoff=1.0, fs=500, order=2):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='high', analog=False)
    y = filtfilt(b, a, data)
    return y

In [ ]:
def extract_qeeg_metrics(df, fs):
    raw_signal = np.asarray(df).flatten()

    filtered_signal = butter_highpass_filter(raw_signal, cutoff=1.0, fs=fs)

    freqs, psd = welch(filtered_signal, fs=fs, nperseg=fs*4)

    bands = {
        'Delta': (1.0, 4),
        'Theta': (4, 8),
        'Alpha': (8, 13),
        'Beta': (13, 30)
    }

    metrics = {}
    total_idx = (freqs >= 1.0) & (freqs <= 30)
    total_power = np.trapezoid(psd[total_idx], freqs[total_idx])

    for band, (low, high) in bands.items():
        idx = (freqs >= low) & (freqs <= high)
        abs_pwr = np.trapezoid(psd[idx], freqs[idx])
        metrics[f'Abs_{band}'] = abs_pwr
        metrics[f'Rel_{band}'] = (abs_pwr / total_power) * 100

    metrics['Alpha_Beta_Ratio'] = metrics['Rel_Alpha'] / metrics['Rel_Beta']
    metrics['Theta_Alpha_Ratio'] = metrics['Rel_Theta'] / metrics['Rel_Alpha']
    metrics['Theta_Beta_Ratio'] = metrics['Rel_Theta'] / metrics['Rel_Beta']

    return metrics

In [ ]:
print("Loading Kaggle CSV file...")

official_channels = [
    'Fp1', 'Fp2', 'F3', 'F4', 'F7', 'F8', 'T3', 'T4', 'C3',
    'C4', 'T5', 'T6', 'P3', 'P4', 'O1', 'O2', 'Fz', 'Cz', 'Pz'
]

df_raw = pd.read_csv("s00.csv", header=None, names=official_channels)

df_raw.columns = df_raw.columns.str.strip()


target_channel = 'F4'

if target_channel in df_raw.columns:
    print(f"Successfully located target channel: {target_channel}")
else:
    print(f"Warning: {target_channel} not found. Available keys: {list(df_raw.columns)}")
    target_channel = df_raw.columns[0]

Loading Kaggle CSV file...
Successfully located target channel: F4


In [ ]:
midpoint = len(df_raw) // 2

baseline_signal = df_raw[target_channel].iloc[:midpoint].values
task_signal = df_raw[target_channel].iloc[midpoint:].values


baseline_results = extract_qeeg_metrics(baseline_signal, fs=500)
task_results = extract_qeeg_metrics(task_signal, fs=500)

comparison_df = pd.DataFrame([baseline_results, task_results],
                             index=['Baseline (Rest)', 'Mental Arithmetic Task']).T

print("\n==============================================")
print(f"         qEEG METRICS COMPARISON FOR {target_channel}           ")
print("==============================================")
print(comparison_df.round(4))
print("==============================================")

diff_alpha_beta = task_results['Alpha_Beta_Ratio'] / baseline_results['Alpha_Beta_Ratio']
diff_theta_alpha = task_results['Theta_Alpha_Ratio'] / baseline_results['Theta_Alpha_Ratio']
diff_theta_beta = task_results['Theta_Beta_Ratio'] / baseline_results['Theta_Beta_Ratio']

print(f"Alpha/Beta: {diff_alpha_beta:.2f}x higher during task.")
print(f"Theta/Alpha: {diff_theta_alpha:.2f}x higher during task.")
print(f"Theta/Beta: {diff_theta_beta:.2f}x higher during task.")


         qEEG METRICS COMPARISON FOR F4           
                   Baseline (Rest)  Mental Arithmetic Task
Abs_Delta                  30.6209                 33.2099
Rel_Delta                  24.9916                 27.0902
Abs_Theta                  16.4679                 16.4264
Rel_Theta                  13.4405                 13.3994
Abs_Alpha                  56.3259                 54.4917
Rel_Alpha                  45.9710                 44.4504
Abs_Beta                   19.1100                 18.4620
Rel_Beta                   15.5968                 15.0599
Alpha_Beta_Ratio            2.9475                  2.9516
Theta_Alpha_Ratio           0.2924                  0.3014
Theta_Beta_Ratio            0.8617                  0.8897
Alpha/Beta: 1.00x higher during task.
Theta/Alpha: 1.03x higher during task.
Theta/Beta: 1.03x higher during task.
